# HW5

### Generate data and Polynomial

In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error

# Generate data
np.random.seed(42) 
X_original = np.random.rand(100, 2) 
x1 = X_original[:, 0]
x2 = X_original[:, 1]

def polynomial(x1, x2):
    return 4 * x1**2 + 5 * x2**2 - 2 * x1 * x2 + 3 * x1 - 6 * x2

y = polynomial(x1, x2)

poly = PolynomialFeatures(degree=2, include_bias=True)
X_poly = poly.fit_transform(X_original)

print(f"X_poly shape: {X_poly.shape}")
print(f"y shape: {y.shape}")

X_poly shape: (100, 6)
y shape: (100,)


### Gradient Descent Methods

In [4]:
# GD
def polynomial_regression_gradient_descent(X, y, lr=0.01, n_iter=1000):
    m, n = X.shape
    theta = np.zeros(n)
    for _ in range(n_iter):
        grad = X.T.dot(X.dot(theta) - y) / m
        theta -= lr * grad
    return theta

# SGD
def polynomial_regression_SGD(X, y, lr=0.01, n_iter=1000):
    m, n = X.shape
    theta = np.zeros(n)
    for _ in range(n_iter):
        for i in range(m):
            idx = np.random.randint(m)
            xi = X[idx:idx+1]
            yi = y[idx]
            grad = xi.T.dot(xi.dot(theta) - yi)
            theta -= lr * grad
    return theta

# RMSProp
def polynomial_regression_rmsprop(X, y, lr=0.01, n_iter=1000, decay=0.9, eps=1e-8):
    m, n = X.shape
    theta = np.zeros(n)
    cache = np.zeros(n)
    for _ in range(n_iter):
        for i in range(m):
            xi = X[i:i+1]
            yi = y[i]
            grad = xi.T.dot(xi.dot(theta) - yi)
            cache = decay * cache + (1 - decay) * grad**2
            theta -= lr * grad / (np.sqrt(cache) + eps)
    return theta

# Adam
def polynomial_regression_adam(X, y, lr=0.01, n_iter=1000, beta1=0.9, beta2=0.999, eps=1e-8):
    m, n = X.shape
    theta = np.zeros(n)
    m_t = np.zeros(n)
    v_t = np.zeros(n)
    t = 0
    for _ in range(n_iter):
        for i in range(m):
            t += 1
            xi = X[i:i+1]
            yi = y[i]
            grad = xi.T.dot(xi.dot(theta) - yi)
            m_t = beta1 * m_t + (1 - beta1) * grad
            v_t = beta2 * v_t + (1 - beta2) * grad**2
            m_hat = m_t / (1 - beta1**t)
            v_hat = v_t / (1 - beta2**t)
            theta -= lr * m_hat / (np.sqrt(v_hat) + eps)
    return theta

# Nadam
def polynomial_regression_nadam(X, y, lr=0.01, n_iter=1000, beta1=0.9, beta2=0.999, eps=1e-8):
    m, n = X.shape
    theta = np.zeros(n)
    m_t = np.zeros(n)
    v_t = np.zeros(n)
    t = 0
    for _ in range(n_iter):
        for i in range(m):
            t += 1
            xi = X[i:i+1]
            yi = y[i]
            grad = xi.T.dot(xi.dot(theta) - yi)
            m_t = beta1 * m_t + (1 - beta1) * grad
            v_t = beta2 * v_t + (1 - beta2) * grad**2
            m_hat = m_t / (1 - beta1**t)
            v_hat = v_t / (1 - beta2**t)
            theta -= lr * (beta1 * m_hat + (1 - beta1) * grad) / (np.sqrt(v_hat) + eps)
    return theta

### Calc exucution time

In [5]:
print("Timing Gradient Descent:")
%timeit polynomial_regression_gradient_descent(X_poly, y, lr=0.01, n_iter=1000)

print("\nTiming SGD:")
%timeit polynomial_regression_SGD(X_poly, y, lr=0.01, n_iter=1000)

print("\nTiming RMSProp:")
%timeit polynomial_regression_rmsprop(X_poly, y, lr=0.01, n_iter=1000)

print("\nTiming Adam:")
%timeit polynomial_regression_adam(X_poly, y, lr=0.01, n_iter=1000)

print("\nTiming Nadam:")
%timeit polynomial_regression_nadam(X_poly, y, lr=0.01, n_iter=1000)

Timing Gradient Descent:
1.41 ms ± 17.3 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)

Timing SGD:
190 ms ± 1.29 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)

Timing RMSProp:
280 ms ± 18.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)

Timing Adam:
398 ms ± 2.98 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)

Timing Nadam:
511 ms ± 66.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


### Find optimal number of iterations

In [8]:
iterations_to_test = [100, 300, 1000]

methods = {
    "GD": polynomial_regression_gradient_descent,
    "SGD": polynomial_regression_SGD,
    "RMSProp": polynomial_regression_rmsprop,
    "Adam": polynomial_regression_adam,
    "Nadam": polynomial_regression_nadam
}

for n in iterations_to_test:
    print(f"--- Testing {n} iterations ---")
    for name, method in methods.items():
        theta = method(X_poly, y, lr=0.01, n_iter=n)
        y_pred = X_poly.dot(theta)
        mse = mean_squared_error(y, y_pred)
        print(f"{name} MSE: {mse:.4f}")
    print("\n")

--- Testing 100 iterations ---
GD MSE: 2.3303
SGD MSE: 0.1038
RMSProp MSE: 0.0044
Adam MSE: 0.0055
Nadam MSE: 0.0055


--- Testing 300 iterations ---
GD MSE: 1.2605
SGD MSE: 0.0317
RMSProp MSE: 0.0012
Adam MSE: 0.0000
Nadam MSE: 0.0000


--- Testing 1000 iterations ---
GD MSE: 0.3331
SGD MSE: 0.0019
RMSProp MSE: 0.0012
Adam MSE: 0.0000
Nadam MSE: 0.0000




### Conslusions

**1. Optimal Number of Iterations:**
* Based on the experiment, **Adam** and **Nadam** are very fast to learn. Their optimal number of iterations is around **200-300**, after which the error (MSE) stops dropping significantly.
* Basic **GD** and **RMSProp** need much more time to learn. Their optimal number is **1000 or more** iterations to reach the same low error.

**2. Computational Efficiency:**
* **Time per iteration:** Standard Gradient Descent (GD) is the fastest method per single run because it uses vectorized matrix operations. SGD, Adam, and Nadam take more time per run because they use loops for each row of data.
* **Overall efficiency:** **Adam** and **Nadam** are the most efficient. Even though one loop takes slightly longer, they require fewer iterations to find the correct weights and achieve the lowest MSE. SGD is fast but too noisy, and standard GD requires too many iterations.